# Diagnosis multiplex hyperedges selection: cliques expansion (POLY)

In [1]:
import pandas as pd
import numpy as np
import hypernetx as hnx
import networkx as nx
import igraph as ig
import matplotlib.pyplot as plt
from itertools import combinations
from math import comb, ceil  # binomial coefficient in Python >=3.8
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Sequence, Tuple
from scipy.special import erf, erfinv
from scipy.stats import rankdata, norm, multivariate_normal
from scipy.optimize import minimize_scalar
import time
from joblib import Parallel, delayed
from statsmodels.stats.multitest import multipletests
import json
import pickle
from pathlib import Path

import os
import sys

In [2]:
# get parent directory
parent_dir = os.path.abspath("..")
sys.path.append(parent_dir)

import preprocessing.candidate_multiplets as cand_mult

import optim_help.O_Information as o_infoptim
import optim_help.paral_perm_bootstrap as pp_bootopt
import optim_help.BCa_bootstrap as bca_bootopt
import optim_help.mult_sel_minimal as sel

import optim_help.plots.o_info_bootstrap_plots as o_plots
import optim_help.plots.BCa_plots as bca_plots

import optim_help.utilities as ut

C:\Users\utente\anaconda3\envs\thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## LOADING DATA

### Data_frames per diagnosis

- **EDI_diags** : the datasets dictionary, where a dataframe of raw data (items as columns, subjects as rows) is associated with the specific diagnosis.

In [3]:
data_path = '../../DATA/EDI_DIAG' # contains csv files for each diagnoses
EDI_diags_paths = [
    data_path + '/' + f
    for f in os.listdir(data_path)
    if f.lower().endswith('.csv') and os.path.isfile(os.path.join(data_path, f))
]

In [4]:
EDI_diags = {}
for path in EDI_diags_paths:
    EDI_diags[Path(path).stem] = ut.preproc_df(path)

In [5]:
EDI_diags.keys()

dict_keys(['ANBP', 'ANR', 'BED_OSFED', 'BN'])

In [6]:
EDI_diags['ANBP'].describe(include='all')

,1,2,3,4,5,6,7,8,9,10,...,82,83,84,85,86,87,88,89,90,91
count,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,...,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000
mean,2.683398,2.598456,1.474903,1.366795,1.254826,1.135135,2.683398,2.073359,2.447876,2.223938,...,1.243243,1.791506,2.274131,2.003861,2.030888,2.054054,1.046332,0.984556,0.911197,2.559846
std,1.291052,1.403538,1.460841,1.381150,1.376972,1.386913,1.452128,1.480632,1.606728,1.387949,...,1.317261,1.376352,1.394253,1.339611,1.465036,1.262524,1.190147,1.181147,1.246635,1.200488
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,1.500000,0.000000,0.000000,0.000000,0.000000,2.000000,1.000000,1.000000,1.000000,...,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,2.000000
50%,3.000000,3.000000,1.000000,1.000000,1.000000,1.000000,3.000000,2.000000,3.000000,2.000000,...,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.000000,0.000000,0.000000,3.000000
75%,4.000000,4.000000,3.000000,3.000000,2.000000,2.000000,4.000000,3.000000,4.000000,4.000000,...,2.000000,3.000000,4.000000,3.000000,3.000000,3.000000,2.000000,2.000000,2.000000,3.000000
max,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,...,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000


### Combinatorial explosion: a problem to be addressed. 

In [7]:
elements = list(range(1, 92))
sizes = [3, 4, 5]

# Build dict
multiplets_dict = {k: list(combinations(elements, k)) for k in sizes}
for i, l in enumerate([len(L) for L in multiplets_dict.values()]):
    print('Size', sizes[i], ':', l, 'possible combinations.')

# 3 121485    1140  (20 items)
# 4 2672670   4845  (20 items)
# 5 46504458  15504 (20 items)

Size 3 : 121485 possible combinations.
Size 4 : 2672670 possible combinations.
Size 5 : 46504458 possible combinations.


### Graphs, polychoric matrices per diagnosis

Create **dictionaries** containing all the necessary data for further computations:

- **graphs_diags** : a nx graph is associated with the diagnosis;
    - graph is loaded from **graphml** object previously computed in R: graph_from_adjacency_matrix based on **polychoric correlation** and **EBICglasso**;
- **poly_matrices_diags** : the polychoric correlation matrix which the graph is based on is associated with the diagnosis.

In [8]:
# GRAPHS -------------------------------------------------
# C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\DATA\GRAPHS_DIAG\POLY_graphs
graph_data_path = '../../DATA/GRAPHS_DIAG/POLY_graphs'

graphs_diags_paths = [
    graph_data_path + '/' + f
    for f in os.listdir(graph_data_path)]

graphs_diags = {}
for path in graphs_diags_paths:
    G_polychor = nx.read_graphml(path)
    # rename the nodes to have the item index as name
    mapping = {name: str(int(name.replace("n", "")) + 1) for name in G_polychor.nodes()}
    key = Path(path).stem.split('_')[0]
    # manually correct the BED_OSFED key
    if key == 'BED':
        key = 'BED_OSFED'
    graphs_diags[key] = nx.relabel_nodes(G_polychor, mapping) 
    
print(graphs_diags.keys())

dict_keys(['ANBP', 'ANR', 'BED_OSFED', 'BN'])


In [9]:
print(graphs_diags['ANR'].nodes)
print(len(graphs_diags['ANR'].edges))

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91']
759


In [10]:
# POLYCHORIC MATRICES ------------------------------------
poly_matrices_data_path = '../../DATA/GRAPHS_DIAG/POLY_mat'

poly_matrices_paths = [
    poly_matrices_data_path + '/' + f
    for f in os.listdir(poly_matrices_data_path)
    if f.lower().endswith('.csv') and os.path.isfile(os.path.join(poly_matrices_data_path, f))
]

poly_matrices_diags = {}
for path in poly_matrices_paths:
    key = Path(path).stem.split('_')[0]
    # manually correct the BED_OSFED key
    if key == 'BED':
        key = 'BED_OSFED'
    poly_matrices_diags[key] = cand_mult.load_correlation_matrix_R(path)
    
print(poly_matrices_diags.keys())

dict_keys(['ANBP', 'ANR', 'BED_OSFED', 'BN'])


In [11]:
poly_matrices_diags['ANBP'].shape

(91, 91)

## Multiplets Selection

### Assumptions on Graphs

- Graphs are connected and undirected
- Weights can be negative
- Spinglass community detection algorithm can handle negative weights and works well with small graphs


### Goal

Given a set of items (Likert variables), we need to select candidate multiplets that are likely to contain **higher-order dependencies**, so they can be statistically validated with the **Ω-information pipeline**. This selection step is necessary to avoid combinatorial explosion.

### Procedure — Stage 1: Pre-selection

> **Paper cited:** Contisciani, M., Battiston, F., & De Bacco, C. (2022). *Inference of hyperedges and overlapping communities in hypergraphs.* Nature Communications, 13, 7229. https://doi.org/10.1038/s41467-022-34714-7

#### Step 1 — Community Detection and Membership Matrix $U$

Communities are detected from the dyadic item network via **Spinglass** and encoded in a hard membership matrix $U$, which serves as a structural prior for candidate multiplet quality.

- Detect Spinglass communities from the weighted graph
- Convert community assignments to a matrix:

$$U \in \{0,1\}^{N \times K}$$

where $u_i$ is the **one-hot membership vector** for item $i$, and $K$ is the number of detected communities.

#### Step 2 — Inter-community Affinity Matrix $W$

The affinity matrix $W \in \mathbb{R}^{K \times K}$ is estimated from the polychoric/correlation matrix if provided, otherwise from graph edge weights:

$$W_{kl} = \frac{\sum_{i \in \mathcal{C}_k,\ j \in \mathcal{C}_l,\ i < j} |R_{ij}|}{|\{(i,j) : i \in \mathcal{C}_k,\ j \in \mathcal{C}_l,\ i < j\}|}$$

where $R_{ij}$ is the correlation between items $i$ and $j$, and $\mathcal{C}_k$ is the set of items in community $k$.

- **Diagonal** $W_{kk}$: intra-community cohesion
- **Off-diagonal** $W_{kl}$: inter-community affinity — how much communities $k$ and $l$ co-move
- High values indicate strongly coupled communities
- A small ridge $\varepsilon$ is added for numerical stability: $W \leftarrow W + \varepsilon$

#### Step 3 — Seed Generation and Scoring

Candidate multiplets are seeded from **maximal cliques** of size $d \in [d_{\min}, d_{\max}]$, extracted via the Bron-Kerbosch algorithm. Cliques larger than $d_{\max}$ are decomposed into all $k$-subsets within the allowed range, ensuring coverage at multiple scales.

Each candidate hyperedge $e$ is ranked with a relaxed **Contisciani-style score**:

$$S(e) = \kappa(|e|) \sum_{i < j \in e} u_i^\top W u_j$$

Since $u_i$ and $u_j$ are one-hot vectors, this reduces to:

$$u_i^\top W u_j = W_{c(i),\ c(j)}$$

which captures the affinity between the communities of items $i$ and $j$.

The **size penalty** $\kappa$ (default: factorial) is:

$$\kappa(|e|) = \frac{1}{|e|!}$$

This penalty:
- strongly discourages large multiplets, favouring small coherent groups
- matches the hyperedge size prior implicit in Contisciani et al. (2022)

A multiplet scores **high** if:
- its items belong to community pairs with strong affinity (high $W_{kl}$)
- its size is not too large (penalized by $\kappa$)

> The score acts as a **structural prior**: *"Is this group supported by the mesoscale community structure?"*

**Relation to Contisciani et al. (2022).** The original model uses a multiplicative score:

$$\lambda(e) = \sum_k w_{d,k} \prod_{i \in e} u_{ik}$$

which requires all nodes in $e$ to share a common community to yield a non-zero contribution. We adopt a relaxed **additive pairwise** formulation for two reasons: (1) the score serves as a pre-filter to produce a broad but plausible candidate pool, not to infer community structure directly; (2) final hyperedge selection is delegated to Ω-information, which can detect higher-order dependencies spanning multiple communities — candidates that a purely intra-community criterion would suppress prematurely.

#### Step 4 — Greedy Expansion

The highest-scoring seeds are **greedily expanded** by adding neighboring nodes, always starting from the original seed (never from derived candidates, to avoid quality drift and combinatorial explosion).

For each seed of size $d$, the algorithm tries to add $k = 1, 2, \ldots, d_{\max} - d$ nodes from the seed's neighborhood, accepting the expansion only if:

$$S(e_{\text{expanded}}) \geq S(e_{\text{seed}}) \cdot (1 + \delta)$$

where $\delta$ is a minimum relative gain threshold (default: $\delta = 0.05$, i.e. 5%).

This produces, for example:
- size-3 seeds → size-4 and size-5 derived candidates
- size-4 seeds → size-5 derived candidates
- size-5 seeds → kept as-is

Only the **top $m$ best expansions** per (seed, expansion size) pair are retained, to control branching. When the number of possible neighbor subsets exceeds a threshold, random sampling is used instead of full enumeration.

## Summary

The candidate-multiplet procedure begins by extracting the mesoscale structure of the item network. Communities are detected from the dyadic graph and encoded in a hard membership matrix $U$, while an inter-community affinity matrix $W$ is estimated either from polychoric correlations or graph edge weights. These two structural priors capture how strongly different item communities cohere or interact.

Using this information, the algorithm seeds candidate groups by identifying all maximal cliques in the item graph within a chosen size range; large cliques are decomposed so that all relevant subgroups are considered. Each seed is scored using a Contisciani-style hyperedge score, which rewards sets supported by strong within- and between-community affinities while penalizing excessively large groups through the $\kappa$ term.

The highest-scoring seeds are subsequently expanded greedily — always from the original seed — adding neighbors that improve the score by a minimum relative gain, thus capturing near-cliques and other dense, well-supported subsets that may not be perfect cliques. Finally, all candidates are organized by size and returned as structured multiplet sets, ready to be evaluated with the Ω-information bootstrap.

In [12]:
out_dir = "POLY_candidates"            
os.makedirs(out_dir, exist_ok=True)

In [20]:
cfg = cand_mult.CandidateConfig(
    k_min=3,
    k_max=5,
    max_initial=None,  # limit if needed
    kappa_mode="factorial",  # stronger downweight for large sets
    expand_max_size=5,  # allow growth beyond cliques 
    expand_top_per_seed=1,  # keep expansions under control
    expand_min_gain_ratio=0.05,  # +5% score improvement to accept
    top_per_size= None, # None or X to keep top X per size; tune as needed
    expand=True # def is True, set according available resources
)

multiplets_diags = {}
for diagnosis in graphs_diags.keys():
    print(diagnosis)
    start = time.perf_counter()
    
    G_nx = graphs_diags[diagnosis]
    G_ig = ig.Graph.from_networkx(G_nx)  
    weights = G_ig.es["weight"] if "weight" in G_ig.es.attributes() else None

    R = poly_matrices_diags[diagnosis]
    # passing R to function
    multiplets_diags[diagnosis] = cand_mult.build_candidate_multiplets(G_ig, R=R, cfg=cfg)
    
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")
    
    fname = f"{diagnosis}_candidates.pkl"
    fpath = os.path.join(out_dir, fname)
    with open(fpath, "wb") as f:
        pickle.dump(multiplets_diags[diagnosis], f, protocol=pickle.HIGHEST_PROTOCOL)

ANBP
  original candidates   : 689
Expanding original candidates...

── expand_candidates summary ──────────────────
  original candidates   : 689
  new candidates added  : 931
  total pool size       : 1620
  skipped (seen)        : 1170
  skipped (score)       : 0
  seeds with sampling   : 0
───────────────────────────────────────────────

Runtime: 9.806329 s
ANR
  original candidates   : 764
Expanding original candidates...

── expand_candidates summary ──────────────────
  original candidates   : 764
  new candidates added  : 1019
  total pool size       : 1783
  skipped (seen)        : 1440
  skipped (score)       : 0
  seeds with sampling   : 0
───────────────────────────────────────────────

Runtime: 10.547433 s
BED_OSFED
  original candidates   : 581
Expanding original candidates...

── expand_candidates summary ──────────────────
  original candidates   : 581
  new candidates added  : 762
  total pool size       : 1343
  skipped (seen)        : 798
  skipped (score)       : 0


In [22]:
for diag in multiplets_diags:
    print_ = diag
    tot = 0
    for order in multiplets_diags[diag]:
        tot += len(multiplets_diags[diag][order])
        print_ = print_ + ', ' + str(order) + ':' + str(len(multiplets_diags[diag][order]))
    print(print_ + ', TOTAL is', str(tot))

ANBP, 5:689, 4:602, 3:329, TOTAL is 1620
ANR, 5:764, 4:642, 3:377, TOTAL is 1783
BED_OSFED, 5:581, 4:517, 3:245, TOTAL is 1343
BN, 5:578, 4:513, 3:265, TOTAL is 1356


####  Compute $\Omega$-information for each multiplet to filter the pre-selected multiplets.

In [23]:
presel_mults_diags = {}
for diagnosis, multiplets in multiplets_diags.items():
    start = time.perf_counter()
    # multiplets is a dict such as: order: multiplets df
    presel_mults_diags[diagnosis] = pp_bootopt.filter_multiplets_by_omega(multiplets_dict = multiplets,
                                                                             X = EDI_diags[diagnosis],
                                                                             threshold=0.1)
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s for {diagnosis}")

Runtime: 3.738761 s for ANBP
Runtime: 4.278381 s for ANR
Runtime: 2.668002 s for BED_OSFED
Runtime: 3.470984 s for BN


In [24]:
for diag in presel_mults_diags:
    print_ = diag
    tot = 0
    for order in presel_mults_diags[diag]:
        tot += len(presel_mults_diags[diag][order])
        print_ = print_ + ', ' + str(order) + ':' + str(len(presel_mults_diags[diag][order]))
    print(print_ + ', TOTAL is', str(tot))

ANBP, 5:683, 4:593, 3:177, TOTAL is 1453
ANR, 5:737, 4:599, 3:138, TOTAL is 1474
BED_OSFED, 5:550, 4:497, 3:178, TOTAL is 1225
BN, 5:575, 4:505, 3:79, TOTAL is 1159


In [25]:
multiplets_diags = presel_mults_diags

## Procedure - Stage 2: Selection

Given a multivariate system X = {X₁, X₂, ..., Xₙ} (representing EDI-3 items), we construct a multiplex hypergraph based on statistically validated higher-order interactions. The global Ω(X₁, ..., Xₙ) summarizes the balance between redundancy and synergy, but can mask localized effects. To identify statistically significant interactions, we evaluate Ω-information on candidate multiplets of order k ∈ {3, 4, 5} through a three-stage validation pipeline, following Marinazzo et al. (2024).

---

### Entropy-Based Ω-Information Computation

**Definition:**

For a set of variables X = {X₁, X₂, ..., Xₖ}, the Ω-information quantifies higher-order interactions:

Ω(X₁, X₂, ..., Xₖ) = Σᵢ H(Xᵢ) - H(X₁, X₂, ..., Xₖ)

Where:
- H(Xᵢ) = Shannon entropy of single variable
- H(X₁, X₂, ..., Xₖ) = Joint entropy of all variables
- Ω > 0: Variables share redundant information
- Ω < 0: Variables interact synergistically (information emerges from combination)

**Computation Method:**

1. **Empirical probability estimation:**
   - Count observed frequencies of each category combination
   - Compute probabilities: P(x₁, x₂, ..., xₖ) = count / n_samples

2. **Entropy calculation:**
   - H(X) = -Σₓ P(x) log P(x)  [typically log₂ or log_e]
   - Applied to marginal distributions for H(Xᵢ)
   - Applied to joint distribution for H(X₁, ..., Xₖ)

3. **Why entropy-based?**
   - Works directly on observed discrete variables (Likert responses)
   - No distributional assumptions required
   - Captures "What is the multivariate interaction in the observed ordinal data?"
   - Computationally efficient for discrete data

---

### Part 1: Significance Testing via Permutation Test

**Purpose:** Identify multiplets with statistically significant Ω-information values.

**Procedure:**

1. **Compute Ω-information** for all candidate multiplets of order k = 3, 4, 5:
   - Ω > 0: Redundancy (overlapping information across variables)
   - Ω < 0: Synergy (emergent, non-decomposable interaction)
   - Ω ≈ 0: No higher-order effect

2. **Column-wise permutation test:**
   - For each multiplet, independently shuffle each variable while preserving marginal distributions
   - This destroys multivariate dependence, yielding a null distribution for Ω
   - Compute two-tailed p-values from null distribution

3. **Multiple comparison correction:**
   - Apply FDR correction via Benjamini–Hochberg method (Benjamini & Hochberg, 1995)
   - Retain multiplets with adjusted p < 0.05

4. **Effect-size filtering:**
   - Apply threshold on absolute Ω: retain only |Ω| ≥ 0.15
   - Rationale: Treat smaller effects as negligible and filter out spurious significance

**Output:** Pruned set of multiplets with significant non-trivial effects

---

#### Implementation Notes

- **Pipeline stages are sequential:** Each stage builds on the validated output of the previous stage
- **Conservative filtering:** Multiple validation criteria ensure low false-positive rate
- **Computational complexity:** O(2^k) for k-multiplet enumeration; practical for k ≤ 5
- **Significance level:** α = 0.05 (with FDR correction in Stage 1)
- **Bootstrap samples:** Recommended n_bootstrap ≥ 1000 for stable CI estimation

---

#### References

- Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate: A practical and powerful approach to multiple testing. *Journal of the Royal Statistical Society: Series B*, 57(1), 289–300.
- DiCiccio, T. J., & Efron, B. (1996). Bootstrap confidence intervals. *Statistical Science*, 11(3), 189–212.
- Efron, B., & Tibshirani, R. J. (1994). *An introduction to the bootstrap*. Chapman and Hall/CRC.
- Efron, B., & Narasimhan, B. (2020). The automatic percentile bootstrap. *arXiv preprint arXiv:2001.03263*.
- Marinazzo, D., et al. (2024). Ω-information and higher-order interactions in complex systems. *[Citation needed]*.


In [27]:
n_iter_boot = 1000

In [30]:
# skip dir creation if it already exist
out_dir = "POLY_part1_perm_boot"  
os.makedirs(out_dir, exist_ok=True)

perm_boot_diags = {}
for diagnosis, multiplets in multiplets_diags.items():
    start = time.perf_counter()
    # multiplets is a dict such as: order: multiplets df
    perm_boot_diags[diagnosis] = pp_bootopt.perm_bootstrap_parallel(multiplets_dict=multiplets,
                                                                    X=EDI_diags[diagnosis], 
                                                                    n_boot=n_iter_boot)
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")
    
    fname = f"{diagnosis}_perm_boot_mults.pkl"
    fpath = os.path.join(out_dir, fname)
    with open(fpath, "wb") as f:
        pickle.dump(perm_boot_diags[diagnosis], f, protocol=pickle.HIGHEST_PROTOCOL)

Families:   0%|          | 0/3 [00:00<?, ?it/s]

  Multiplets | order=5:   0%|0/683 |          | 00:00

  Multiplets | order=5:  25%|171/683 |██▌       | 01:33

  Multiplets | order=5:  50%|342/683 |█████     | 03:01

  Multiplets | order=5:  75%|513/683 |███████▌  | 04:30

  Multiplets | order=5: 100%|683/683 |██████████| 05:58

Families:  33%|███▎      | 1/3 [05:58<11:56, 358.27s/it][A

  Multiplets | order=4:   0%|0/593 |          | 00:00

  Multiplets | order=4:  25%|149/593 |██▌       | 00:58

  Multiplets | order=4:  50%|298/593 |█████     | 01:57

  Multiplets | order=4:  75%|447/593 |███████▌  | 02:55

  Multiplets | order=4: 100%|593/593 |██████████| 03:54

Families:  67%|██████▋   | 2/3 [09:52<04:45, 285.27s/it][A

  Multiplets | order=3:   0%|0/177 |          | 00:00

  Multiplets | order=3:  25%|45/177 |██▌       | 00:12

  Multiplets | order=3:  51%|90/177 |█████     | 00:24

  Multiplets | order=3:  76%|135/177 |███████▋  | 00:38

  Multiplets | order=3: 100%|177/177 |███

Runtime: 643.038612 s


Families:   0%|          | 0/3 [00:00<?, ?it/s]

  Multiplets | order=5:   0%|0/737 |          | 00:00

  Multiplets | order=5:  25%|185/737 |██▌       | 02:03

  Multiplets | order=5:  50%|370/737 |█████     | 04:03

  Multiplets | order=5:  75%|555/737 |███████▌  | 06:04

  Multiplets | order=5: 100%|737/737 |██████████| 08:05

Families:  33%|███▎      | 1/3 [08:05<16:11, 485.60s/it][A

  Multiplets | order=4:   0%|0/599 |          | 00:00

  Multiplets | order=4:  25%|150/599 |██▌       | 01:14

  Multiplets | order=4:  50%|300/599 |█████     | 02:30

  Multiplets | order=4:  75%|450/599 |███████▌  | 03:44

  Multiplets | order=4: 100%|599/599 |██████████| 04:58

Families:  67%|██████▋   | 2/3 [13:03<06:15, 375.40s/it][A

  Multiplets | order=3:   0%|0/138 |          | 00:00

  Multiplets | order=3:  25%|35/138 |██▌       | 00:12

  Multiplets | order=3:  51%|70/138 |█████     | 00:25

  Multiplets | order=3:  76%|105/138 |███████▌  | 00:37

  Multiplets | order=3: 100%|138/138 |███

Runtime: 834.313084 s


Families:   0%|          | 0/3 [00:00<?, ?it/s]

  Multiplets | order=5:   0%|0/550 |          | 00:00

  Multiplets | order=5:  25%|138/550 |██▌       | 01:03

  Multiplets | order=5:  50%|276/550 |█████     | 02:08

  Multiplets | order=5:  75%|414/550 |███████▌  | 03:14

  Multiplets | order=5: 100%|550/550 |██████████| 04:24

Families:  33%|███▎      | 1/3 [04:24<08:49, 264.77s/it][A

  Multiplets | order=4:   0%|0/497 |          | 00:00

  Multiplets | order=4:  25%|125/497 |██▌       | 00:53

  Multiplets | order=4:  50%|250/497 |█████     | 01:40

  Multiplets | order=4:  75%|375/497 |███████▌  | 02:28

  Multiplets | order=4: 100%|497/497 |██████████| 03:10

Families:  67%|██████▋   | 2/3 [07:35<03:40, 220.99s/it][A

  Multiplets | order=3:   0%|0/178 |          | 00:00

  Multiplets | order=3:  25%|45/178 |██▌       | 00:11

  Multiplets | order=3:  51%|90/178 |█████     | 00:23

  Multiplets | order=3:  76%|135/178 |███████▌  | 00:36

  Multiplets | order=3: 100%|178/178 |███

Runtime: 502.890184 s


Families:   0%|          | 0/3 [00:00<?, ?it/s]

  Multiplets | order=5:   0%|0/575 |          | 00:00

  Multiplets | order=5:  25%|144/575 |██▌       | 01:49

  Multiplets | order=5:  50%|288/575 |█████     | 03:37

  Multiplets | order=5:  75%|432/575 |███████▌  | 05:31

  Multiplets | order=5: 100%|575/575 |██████████| 07:19

Families:  33%|███▎      | 1/3 [07:19<14:39, 439.79s/it][A

  Multiplets | order=4:   0%|0/505 |          | 00:00

  Multiplets | order=4:  25%|127/505 |██▌       | 01:11

  Multiplets | order=4:  50%|254/505 |█████     | 02:22

  Multiplets | order=4:  75%|381/505 |███████▌  | 03:34

  Multiplets | order=4: 100%|505/505 |██████████| 04:43

Families:  67%|██████▋   | 2/3 [12:03<05:48, 348.08s/it][A

  Multiplets | order=3:   0%|0/79 |          | 00:00

  Multiplets | order=3:  25%|20/79 |██▌       | 00:07

  Multiplets | order=3:  51%|40/79 |█████     | 00:15

  Multiplets | order=3:  76%|60/79 |███████▌  | 00:23

  Multiplets | order=3: 100%|79/79 |██████████

Runtime: 755.111883 s


### Stage 2, part 1: plots.

In [31]:
base = 'POLY_part1_plots/'
perm_summary_tables = {}
for diag, out in perm_boot_diags.items():
    perm_summary_tables[diag] = o_plots.save_all_figures(out, output_dir=base + diag + "_figures")

Saved -> POLY_part1_plots/ANBP_figures\fig1_null_distributions.pdf
Saved -> POLY_part1_plots/ANBP_figures\fig2_omega_scatter.pdf
Saved -> POLY_part1_plots/ANBP_figures\fig3_omega_violin.pdf


C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:600: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  def _save(fig: plt.Figure, name: str) -> None:
C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:600: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  def _save(fig: plt.Figure, name: str) -> None:


Saved -> POLY_part1_plots/ANBP_figures\fig4_volcano.pdf
Supplementary table saved → POLY_part1_plots/ANBP_figures\supplementary_table.csv  (1453 multiplets)
Saved -> POLY_part1_plots/ANR_figures\fig1_null_distributions.pdf
Saved -> POLY_part1_plots/ANR_figures\fig2_omega_scatter.pdf
Saved -> POLY_part1_plots/ANR_figures\fig3_omega_violin.pdf


C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:600: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  def _save(fig: plt.Figure, name: str) -> None:
C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:600: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  def _save(fig: plt.Figure, name: str) -> None:


Saved -> POLY_part1_plots/ANR_figures\fig4_volcano.pdf
Supplementary table saved → POLY_part1_plots/ANR_figures\supplementary_table.csv  (1474 multiplets)
Saved -> POLY_part1_plots/BED_OSFED_figures\fig1_null_distributions.pdf
Saved -> POLY_part1_plots/BED_OSFED_figures\fig2_omega_scatter.pdf
Saved -> POLY_part1_plots/BED_OSFED_figures\fig3_omega_violin.pdf


C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:600: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  def _save(fig: plt.Figure, name: str) -> None:
C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:600: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  def _save(fig: plt.Figure, name: str) -> None:


Saved -> POLY_part1_plots/BED_OSFED_figures\fig4_volcano.pdf
Supplementary table saved → POLY_part1_plots/BED_OSFED_figures\supplementary_table.csv  (1225 multiplets)
Saved -> POLY_part1_plots/BN_figures\fig1_null_distributions.pdf
Saved -> POLY_part1_plots/BN_figures\fig2_omega_scatter.pdf
Saved -> POLY_part1_plots/BN_figures\fig3_omega_violin.pdf


C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:600: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  def _save(fig: plt.Figure, name: str) -> None:
C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:600: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  def _save(fig: plt.Figure, name: str) -> None:


Saved -> POLY_part1_plots/BN_figures\fig4_volcano.pdf
Supplementary table saved → POLY_part1_plots/BN_figures\supplementary_table.csv  (1159 multiplets)


Load pickle files.

In [32]:
# Load
with open("POLY_part1_perm_boot/ANBP_perm_boot_mults.pkl", "rb") as f:
    ANBP_perm_boot = pickle.load(f)

with open("POLY_part1_perm_boot/ANR_perm_boot_mults.pkl", "rb") as f:
    ANR_perm_boot = pickle.load(f)

with open("POLY_part1_perm_boot/BED_OSFED_perm_boot_mults.pkl", "rb") as f:
    BED_OSFED_perm_boot = pickle.load(f)

with open("POLY_part1_perm_boot/BN_perm_boot_mults.pkl", "rb") as f:
    BN_perm_boot = pickle.load(f)


# rebuild general dictionary
perm_boot_diags = {
    'ANBP' : ANBP_perm_boot,
    'ANR' : ANR_perm_boot,
    'BED_OSFED' : BED_OSFED_perm_boot,
    'BN' : BN_perm_boot,
}

### "Shifted" permutation-based null distribution
In the empirical analyses, the permutation-based null distribution of the $\Omega$-information exhibits a noticeable shift toward negative values (often around $-2$) rather than being centered at zero, as expected from synthetic (benchmark) data. This may originate from a property of applying column-wise permutation tests to real-world categorical data such as Likert-type survey responses. The permutation scheme enforces a null hypothesis of complete statistical independence across variables while **preserving the empirical marginal distributions** of each item. Because real psychometric items typically show unbalanced category frequencies, heterogeneous entropy across items, the expected value of multivariate interaction measures under this form of independence is generally not equal to zero. 

In particular, marginal imbalance inflates both the joint entropy and the exclusion entropies in a direction that systematically
drives the expected O-information into the negative domain. This phenomenon is well aligned with theoretical insights on interaction information and co-information for discrete variables (McGill1954, Bell2003), which show that non-uniform marginals induce baseline shifts even under perfect independence. In contrast, synthetic datasets used for validation usually employ balanced, exchangeable categories, for which the same permutation procedure naturally yields a null distribution centered near zero. 

For applied inference, the appropriate response is therefore to employ a bias-aware testing strategy that evaluates the statistic relative to its empirical null mean (use the two-sided-centered o auto alternative), which assesses $\lvert \Omega_{\mathrm{obs}} - \mathbb{E}[\Omega_{\mathrm{null}}] \rvert$ and thereby provides valid significance testing even when the permutation null is shifted or skewed.

In [33]:
ANBP_perm_boot[3]

{(1, 6, 16): (-0.14771069730976194,
  0.967032967032967,
  array([-0.14512407, -0.11110028, -0.14353703, -0.16173715, -0.1381046 ,
         -0.14211688, -0.11427459, -0.14227955, -0.1489767 , -0.12445348,
         -0.12419961, -0.11348387, -0.17616312, -0.1066426 , -0.1299044 ,
         -0.15575377, -0.14206722, -0.12873968, -0.17543675, -0.14178759,
         -0.11388524, -0.16455285, -0.08935584, -0.14105018, -0.18466994,
         -0.17484292, -0.10373849, -0.21040528, -0.11989299, -0.11374233,
         -0.11819659, -0.14482208, -0.19753199, -0.16430773, -0.1268419 ,
         -0.12849957, -0.14509298, -0.12084224, -0.15896698, -0.14261258,
         -0.11706605, -0.08916505, -0.12804469, -0.18822456, -0.19135722,
         -0.11106579, -0.22664771, -0.12791898, -0.14813732, -0.17733757,
         -0.18609535, -0.15468552, -0.1539704 , -0.14545122, -0.12207776,
         -0.10911636, -0.16496842, -0.13692967, -0.10812948, -0.10435742,
         -0.15327065, -0.14055172, -0.15644533, -0.1442

In [34]:
ut.summary(ANBP_perm_boot)[3]

{(1, 6, 16): (-0.14771069730976194, 0.967032967032967),
 (1, 6, 27): (-0.11194522777220861, 0.002997002997002997),
 (1, 6, 42): (-0.1280101513527896, 0.04295704295704296),
 (1, 6, 70): (-0.19085848070205103, 0.9050949050949051),
 (1, 13, 47): (-0.160363939390896, 0.43656343656343655),
 (1, 19, 47): (-0.14530435448372003, 0.6873126873126874),
 (1, 27, 47): (-0.1325969524178916, 0.07892107892107893),
 (1, 42, 47): (-0.12220900512239297, 0.06493506493506493),
 (2, 24, 36): (-0.18108155147178806, 0.3336663336663337),
 (3, 14, 35): (-0.16108637641459822, 0.0989010989010989),
 (3, 35, 39): (-0.10767264119630582, 0.003996003996003996),
 (5, 7, 53): (-0.1935612566339966, 0.8041958041958042),
 (5, 12, 29): (-0.182027888500917, 0.3146853146853147),
 (5, 12, 53): (-0.18228783739092336, 0.5794205794205795),
 (5, 12, 69): (-0.10433014878430846, 0.023976023976023976),
 (5, 29, 43): (-0.20233598015785148, 0.8131868131868132),
 (5, 29, 71): (-0.26841852606937167, 0.000999000999000999),
 (6, 14, 35): (

### Filter multiplets - pvalue wise + REDUNDANCY/SYNERGY check
Filter out the non significant multiplets and the ones whose observed and significant $\Omega$ - information shows zero-interaction (set a **tolerance** for what is zero-interaction: **|$\Omega$| <= omega_threshold**. 

In [43]:
omega_threshold = 0.1
alpha_significance = 0.05

fdr_neut_filtered_mults_diags = {}
filtered_descriptive = {}
for diagnosis, perm_boot in perm_boot_diags.items():
    fdr_neut_filtered_mults_diags[diagnosis] = ut.fdr_neuter_filter(perm_boot, omega_threshold = omega_threshold)
    filtered_descriptive[diagnosis] = ut.fdr_neuter_filter(perm_boot, omega_threshold = 0)

In [44]:
# here are the fildered multiplets to use for further computations
op_multiplets_diags_F = ut.get_multiplets_as_dict_list_F(fdr_neut_filtered_mults_diags)

In [45]:
#len(set(multiplets_diags['BED'][5]) & set(op_multiplets_diags_['BED'][5])) 

The following is useful for **descriptive purposes**: how many multiplets are significant (neuter/synergistic/redudndant)? how many are not?

In [46]:
omega_partitions_diags = {}
for diag, filtered_boot in filtered_descriptive.items():
    omega_partitions_diags[diag] = ut.omega_partition(filtered_boot, alpha = alpha_significance, omega_threshold = omega_threshold)
    
print(ut.describe_omega_partition(omega_partitions_diags))

Diagnosis: ANBP
	 Interation type: syn there are:
	 	 Order: 5 # multiplets: 674
	 	 Order: 4 # multiplets: 528
	 	 Order: 3 # multiplets: 23
	 Interation type: red there are:
	 	 Order: 5 # multiplets: 3
	 	 Order: 4 # multiplets: 4
	 	 Order: 3 # multiplets: 1
	 Interation type: neut there are:
	 	 Order: 5 # multiplets: 0
	 	 Order: 4 # multiplets: 0
	 	 Order: 3 # multiplets: 0
	 Interation type: non_sig there are:
	 	 Order: 5 # multiplets: 0
	 	 Order: 4 # multiplets: 0
	 	 Order: 3 # multiplets: 0
Diagnosis: ANR
	 Interation type: syn there are:
	 	 Order: 5 # multiplets: 669
	 	 Order: 4 # multiplets: 466
	 	 Order: 3 # multiplets: 2
	 Interation type: red there are:
	 	 Order: 5 # multiplets: 15
	 	 Order: 4 # multiplets: 21
	 	 Order: 3 # multiplets: 31
	 Interation type: neut there are:
	 	 Order: 5 # multiplets: 0
	 	 Order: 4 # multiplets: 0
	 	 Order: 3 # multiplets: 0
	 Interation type: non_sig there are:
	 	 Order: 5 # multiplets: 0
	 	 Order: 4 # multiplets: 0
	 	 Orde

### Part 2: Robustness and Stability via Bootstrap

**Purpose:** Validate sign stability and point estimate reliability.

**Procedure:**

1. **Row-wise nonparametric bootstrap resampling:**
   - Resample observations (rows) with replacement n_bootstrap times
   - For each bootstrap sample, recompute Ω-information

2. **Confidence interval computation (BiasCA):**
   - Compute BCa (Bias-Corrected and Accelerated) confidence intervals (DiCiccio & Efron, 1996; Efron & Tibshirani, 1994; Efron & Narasimhan, 2020)
   - Uses jackknife-based acceleration correction
   - Fallback: Percentile confidence intervals if acceleration computation fails

3. **Stability criteria:**
   - **Sign stability:** Retain multiplets whose CI does not include zero
     - Rationale: Ensures consistent classification as redundant or synergistic
   - **Point estimate reliability:** Assess bootstrap distribution for outliers or multimodality
     - Discard if CI is very wide or bootstrap distribution appears unreliable

**Output:** Bootstrap-validated multiplets with stable, reliable effect estimates

#### Compute the bootstrap row wise omega distributions first.

In [48]:
out_dir_sd = "samp_distro_diags_POLY"
os.makedirs(out_dir_sd, exist_ok=True)

bca_samp_distros_diags = {}
for diagnosis, multiplets in op_multiplets_diags_F.items():
    start = time.perf_counter()
    # structure: order: multiple: samp_distro
    ### bca_bootopt.get_boot_stats_parallel is the same as output but handle the parallelism differenty
    bca_samp_distros_diags[diagnosis] = bca_bootopt.get_boot_stats_parallel_chunked(multiplets_dict=multiplets, 
                                                                            X=EDI_diags[diagnosis], 
                                                                            n_boot=n_iter_boot) 
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")
    
    fname = f"{diagnosis}_bca_selected_mults.pkl"
    fpath = os.path.join(out_dir_sd, fname)
    with open(fpath, "wb") as f:
        pickle.dump(bca_samp_distros_diags[diagnosis], f, protocol=pickle.HIGHEST_PROTOCOL)

Families:   0%|          | 0/3 [00:00<?, ?it/s]


Multiplets | family 5:   0%|0/677 |          | 00:00


Multiplets | family 5:  15%|100/677 |█▍        | 00:52


Multiplets | family 5:  30%|200/677 |██▉       | 01:39


Multiplets | family 5:  44%|300/677 |████▍     | 02:24


Multiplets | family 5:  59%|400/677 |█████▉    | 03:09


Multiplets | family 5:  74%|500/677 |███████▍  | 03:54


Multiplets | family 5:  89%|600/677 |████████▊ | 04:39


Multiplets | family 5: 100%|677/677 |██████████| 05:14


Families:  33%|███▎      | 1/3 [05:14<10:28, 314.50s/it]A

  [family 5] bootstrap progress: 677/677 multiplets (100%)





Multiplets | family 4:   0%|0/532 |          | 00:00


Multiplets | family 4:  19%|100/532 |█▉        | 00:36


Multiplets | family 4:  38%|200/532 |███▊      | 01:11


Multiplets | family 4:  56%|300/532 |█████▋    | 01:47


Multiplets | family 4:  75%|400/532 |███████▌  | 02:21


Multiplets | family 4:  94%|500/532 |█████████▍| 02:56


Multiplets | family 4: 100%|532/532 |██████████| 03:07


Families:  67%|██████▋   | 2/3 [08:22<04:00, 240.10s/it]A

  [family 4] bootstrap progress: 532/532 multiplets (100%)





Multiplets | family 3:   0%|0/24 |          | 00:00


Multiplets | family 3: 100%|24/24 |██████████| 00:05




...

  [family 5] bootstrap progress: 563/563 multiplets (100%)



Multiplets | family 4:   0%|0/391 |          | 00:00
Multiplets | family 4:  26%|100/391 |██▌       | 00:46
Multiplets | family 4:  51%|200/391 |█████     | 01:34
Multiplets | family 4:  77%|300/391 |███████▋  | 02:20
Multiplets | family 4: 100%|391/391 |██████████| 03:04
Families:  67%|██████▋   | 2/3 [09:03<04:16, 256.15s/it]A

  [family 4] bootstrap progress: 391/391 multiplets (100%)



Multiplets | family 3:   0%|0/5 |          | 00:00
Multiplets | family 3: 100%|5/5 |██████████| 00:02
                                                        

  [family 3] bootstrap progress: 5/5 multiplets (100%)
Runtime: 545.335277 s


Load pickle files.

In [49]:
# Load
with open("samp_distro_diags_POLY/ANBP_bca_selected_mults.pkl", "rb") as f:
    ANBP_bca_boot = pickle.load(f)

with open("samp_distro_diags_POLY/ANR_bca_selected_mults.pkl", "rb") as f:
    ANR_bca_boot = pickle.load(f)

with open("samp_distro_diags_POLY/BED_OSFED_bca_selected_mults.pkl", "rb") as f:
    BED_OSFED_bca_boot = pickle.load(f)

with open("samp_distro_diags_POLY/BN_bca_selected_mults.pkl", "rb") as f:
    BN_bca_boot = pickle.load(f)


# rebuild general dictionary
bca_samp_distros_diags = {
    'ANBP' : ANBP_bca_boot,
    'ANR' : ANR_bca_boot,
    'BED_OSFED' : BED_OSFED_bca_boot,
    'BN' : BN_bca_boot,
}

#### Assambling information: 
bca_samp_distros_diags contains the emp distros from the bca sampling for each multiplet.
Assemble info from multiplets and sample distributions just computed, by adding to the dict of the fdr survived multiplets info about the BCa sample_distro

In [54]:
BCa_fdr_diags = {}
for diagnosis in bca_samp_distros_diags:
    BCa_fdr_diags[diagnosis] = ut.get_BCa_input_dict_from_fdr(bca_samp_distros_diags[diagnosis],
                                                              fdr_neut_filtered_mults_diags[diagnosis])

BCa_fdr_diags contains the necessary info for computing the BCa (sample_distribution - row wise permutation included)
#### BCa

In [55]:
bca_multiplets_diags = {}
# for diagnosis, multiplets in op_multiplets_diags.items():
for diagnosis, multiplets in BCa_fdr_diags.items():
    start = time.perf_counter()
    # structure: order: multiple: samp_distro
    bca_multiplets_diags[diagnosis] = bca_bootopt.bca_bootstrap_parallel(multiplets_dict=multiplets, 
                                                                            X=EDI_diags[diagnosis])
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")

Families:   0%|          | 0/3 [00:00<?, ?it/s]


[BCa | family 5] starting – 677 multiplets


Families:  33%|███▎      | 1/3 [00:02<00:04,  2.31s/it]

  [BCa | family 5] 677/677 multiplets (100%)

[BCa | family 4] starting – 532 multiplets


Families: 100%|██████████| 3/3 [00:02<00:00,  1.15it/s]


...

  [BCa | family 4] 475/475 multiplets (100%)

[BCa | family 3] starting – 67 multiplets
  [BCa | family 3] 67/67 multiplets (100%)
Runtime: 0.553394 s


Families:   0%|          | 0/3 [00:00<?, ?it/s]


[BCa | family 5] starting – 563 multiplets


Families: 100%|██████████| 3/3 [00:00<00:00,  6.56it/s]

  [BCa | family 5] 563/563 multiplets (100%)

[BCa | family 4] starting – 391 multiplets
  [BCa | family 4] 391/391 multiplets (100%)

[BCa | family 3] starting – 5 multiplets
  [BCa | family 3] 5/5 multiplets (100%)
Runtime: 0.457038 s


### CI filtering
When running the BCa_CI_check procedure, the multiplets with not valid pvalue have already been filtered out.

In [58]:
from contextlib import redirect_stdout

BCa_selected_diags = {}
for diagnosis, multiplets in bca_multiplets_diags.items():
    with open(diagnosis + "_CI_warnings_POLY.txt", "w") as f:
        with redirect_stdout(f):
            BCa_selected_diags[diagnosis] = bca_bootopt.BCa_CI_mults_selection(bca_multiplets_diags[diagnosis])

### Stage 2, part 2: BCa_CI: plots and stats

In [59]:
base = 'POLY_part2_plots/'
n_boot = 1000

for diagnosis in bca_samp_distros_diags:
    out_dir = f"{base}{diagnosis}_figures"
    
    # bootstrap sampling
    # boot_arrays = bootstrap_multiplets_chunked(multiplets_dict=multiplets, X=X, n_boot=500)
    boot_arrays = bca_samp_distros_diags[diagnosis]

    # BCa confidence intervals
    # BCa_boot = bca_bootstrap_parallel(multiplets_dict=..., X=X, alpha=0.05)
    BCa_boot = bca_multiplets_diags[diagnosis]
    
    # selection — keeps only {family: {multiplet: hypothesis}}
    # BCa_selected = BCa_CI_mults_selection(BCa_boot, tol=0.05)
    BCa_selected = BCa_selected_diags[diagnosis]
    
    # re-join with full info for the survivors  
    BCa_boot_sel, boot_arrays_sel = bca_bootopt.retrieve_selected_multiplets_info(
        BCa_selected = BCa_selected,
        BCa_boot_full = BCa_boot,
        boot_arrays_full = boot_arrays,
    )              # {family: {multiplet: (obs, lo, hi, diag)}}

    # pick representative multiplets and plot
    reps = bca_plots.pick_representative_multiplets(BCa_boot_sel, 
                                                    boot_arrays_sel)
    bca_plots.plot_all(
        BCa_boot_all = BCa_boot_sel,
        n_samples = EDI_diags[diagnosis].shape[0],
        n_boot = n_boot,
        representative_multiplets = reps,
        output_dir=out_dir,
    )

  Family 5: 52/677 multiplets retrieved.
  Family 4: 530/532 multiplets retrieved.
  Family 3: 21/24 multiplets retrieved.

Representative multiplets selected from family 4:
  most_typical        multiplet=(1, 27, 45, 55)  Ω=-0.7090  CI=[-0.9114, -0.5208]  margin from 0=0.5208  method=percentile
  most_borderline     multiplet=(8, 21, 51, 60)  Ω=-0.2924  CI=[-0.5591, -0.0056]  margin from 0=0.0056  method=percentile
  most_extreme        multiplet=(25, 33, 40, 51)  Ω=-1.2323  CI=[-1.4444, -0.9570]  margin from 0=0.9570  method=percentile
Generating Figure 1 – Ω dot-and-whisker with BCa CIs ...
  saved → POLY_part2_plots/ANBP_figures\fig1_omega_ci.pdf / .png
Generating Figure 2 – Redundancy / synergy breakdown ...
  saved → POLY_part2_plots/ANBP_figures\fig2_breakdown.pdf / .png
Generating Figure 3 – Bootstrap distribution for multiplet (1, 27, 45, 55) ...
  saved → POLY_part2_plots/ANBP_figures\fig3_boot_dist_fam4_mult1_27_45_55.pdf / .png
Generating Figure 3 – Bootstrap distribution f

Save to pickle files.

In [60]:
out_dir_bca = "bca_selected_POLY"  
os.makedirs(out_dir_bca, exist_ok=True)

for diagnosis, bca_multiplets in BCa_selected_diags.items():
    fname = f"{diagnosis}_bca_selected_mults.pkl"
    fpath = os.path.join(out_dir_bca, fname)
    with open(fpath, "wb") as f:
        pickle.dump(bca_multiplets, f, protocol=pickle.HIGHEST_PROTOCOL)

### Hyperedges selection's results

- fdr_neut_filtered_mults_diags (???)

- bca_selected

In [61]:
# Load
with open("bca_selected_POLY/ANBP_bca_selected_mults.pkl", "rb") as f:
    ANBP_bca_sel = pickle.load(f)

with open("bca_selected_POLY/ANR_bca_selected_mults.pkl", "rb") as f:
    ANR_bca_sel = pickle.load(f)

with open("bca_selected_POLY/BED_OSFED_bca_selected_mults.pkl", "rb") as f:
    BED_OSFED_bca_sel = pickle.load(f)

with open("bca_selected_POLY/BN_bca_selected_mults.pkl", "rb") as f:
    BN_bca_sel = pickle.load(f)


# rebuild general dictionary
BCa_selected_diags = {
    'ANBP' : ANBP_bca_sel,
    'ANR' : ANR_bca_sel,
    'BED_OSFED' : BED_OSFED_bca_sel,
    'BN' : BN_bca_sel
}

How many multiplets have been selected? of which type?

In [62]:
ut.describe_hyperedges(BCa_selected_diags)

Diagnosis: ANBP
	 Order 5:
	 	 Synergy: 51 Redundancy: 1
	 Order 4:
	 	 Synergy: 528 Redundancy: 2
	 Order 3:
	 	 Synergy: 21 Redundancy: 0
Diagnosis: ANR
	 Order 5:
	 	 Synergy: 321 Redundancy: 10
	 Order 4:
	 	 Synergy: 464 Redundancy: 7
	 Order 3:
	 	 Synergy: 2 Redundancy: 18
Diagnosis: BED_OSFED
	 Order 5:
	 	 Synergy: 25 Redundancy: 16
	 Order 4:
	 	 Synergy: 470 Redundancy: 4
	 Order 3:
	 	 Synergy: 55 Redundancy: 0
Diagnosis: BN
	 Order 5:
	 	 Synergy: 288 Redundancy: 0
	 Order 4:
	 	 Synergy: 268 Redundancy: 0
	 Order 3:
	 	 Synergy: 0 Redundancy: 3


#### Compute omega for the survived multiplets.

In [63]:
selected_multiplets_diags = {}
for diagnosis, multiplets in BCa_selected_diags.items():
    start = time.perf_counter()
    selected_multiplets_diags[diagnosis] = pp_bootopt.compute_omega_for_multiplets(multiplets_dict=multiplets, 
                                                                                   X=EDI_diags[diagnosis])
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")

Runtime: 0.941396 s
Runtime: 1.902751 s
Runtime: 0.763786 s
Runtime: 1.308578 s


In [64]:
selected_multiplets_diags['BED_OSFED']

,order,label,omega,node1,node2,node3,node4,node5
0,3,synergy,-0.114467,6,7,8,<NA>,<NA>
1,3,synergy,-0.107667,6,7,85,<NA>,<NA>
2,3,synergy,-0.134632,6,35,76,<NA>,<NA>
3,3,synergy,-0.134829,8,27,44,<NA>,<NA>
4,3,synergy,-0.124578,10,18,20,<NA>,<NA>
...,...,...,...,...,...,...,...,...
565,5,redundancy,0.247899,7,11,16,49,68
566,5,redundancy,0.240815,7,16,32,49,53
567,5,synergy,-0.978040,8,12,21,88,90
568,5,synergy,-0.402055,9,10,20,42,55


In [65]:
# skip dir creation if it already exist
out_dir_sm = "selected_cliques_POLY" 
os.makedirs(out_dir_sm, exist_ok=True)

for diagnosis, df in selected_multiplets_diags.items():
    fname = f"{diagnosis}_selected_df_ce.csv"
    fpath = os.path.join(out_dir_sm, fname)
    df.to_csv(fpath, index=False)

### Part 3: MULTIPLETS' CIs COMPARISON (Hierarchical Pruning)

**Purpose:** Remove multiplets that are explained by their lower-order subsets.

**Procedure:**

1. **Comparison with (k-1)-subsets:**
   - For each k-multiplet M with confidence interval CI(M), extract all (k-1)-subsets
   - For each subset S, retrieve its confidence interval CI(S)

2. **Overlap detection:**
   - Check if CI(M) overlaps with CI(S) for any subset
   - Overlap criterion: max(lo_M, lo_S) ≤ min(hi_M, hi_S)

3. **Filtering rule:**
   - **Discard M if** any (k-1)-subset's CI overlaps with M's CI
   - **Rationale:** Overlapping CIs indicate no statistically significant added higher-order effect beyond lower-order structure (Marinazzo et al., 2024)
   - **Keep M if** no subset CI overlaps (genuine k-order effect)

**Output:** Hierarchically-validated multiplets representing true higher-order interactions.

Get the right structure for further functs.

In [66]:
# pre: BCa_selected_diags['ANBP'][5][(-,-,-,-,-)]: synergy/redundancy
selected_as_list = ut.get_selected_INTER(BCa_selected_diags)

Recompute row wise omega distributions for CIs, then CIs (the whole procedure for selected multiplets) - for Hierchical Pruning.

In [67]:
selected_boot_stats = {}
for diagnosis, mults in selected_as_list.items():
    start = time.perf_counter()
    selected_boot_stats[diagnosis] = bca_bootopt.get_boot_stats_parallel(multiplets_dict=mults, 
                                                                            X=EDI_diags[diagnosis], 
                                                                            n_boot=1000)
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")

Families:   0%|          | 0/3 [00:00<?, ?it/s]
Multiplets | family 5:   0%|          | 0/52 [00:00<?, ?it/s]


  0%|          | 0/52 [00:00<?, ?it/s]


  2%|▏         | 1/52 [00:05<04:39,  5.49s/it]


  6%|▌         | 3/52 [00:05<01:11,  1.46s/it]


 15%|█▌        | 8/52 [00:05<00:18,  2.40it/s]


 21%|██        | 11/52 [00:08<00:27,  1.47it/s]


 27%|██▋       | 14/52 [00:09<00:17,  2.20it/s]


 31%|███       | 16/52 [00:09<00:12,  2.80it/s]


 35%|███▍      | 18/52 [00:12<00:23,  1.47it/s]


 38%|███▊      | 20/52 [00:12<00:16,  1.91it/s]


 42%|████▏     | 22/52 [00:12<00:11,  2.51it/s]


 46%|████▌     | 24/52 [00:13<00:09,  3.00it/s]


 48%|████▊     | 25/52 [00:15<00:20,  1.34it/s]


 52%|█████▏    | 27/52 [00:16<00:13,  1.86it/s]


 54%|█████▍    | 28/52 [00:16<00:13,  1.83it/s]


 60%|█████▉    | 31/52 [00:16<00:07,  2.96it/s]


 62%|██████▏   | 32/52 [00:17<00:07,  2.59it/s]


 63%|██████▎   | 33/52 [00:19<00:12,  1.52it/s]


 65%|██████▌   | 34/52 [00:19<00:10,  1.78it/s]




 25%|██▍       | 131/530 [00:46<02:19,  2.87it/s]


 25%|██▌       | 133/530 [00:46<02:16,  2.90it/s]


 25%|██▌       | 134/530 [00:47<02:18,  2.87it/s]


 25%|██▌       | 135/530 [00:47<02:09,  3.06it/s]


 26%|██▌       | 136/530 [00:47<01:58,  3.32it/s]


 26%|██▌       | 137/530 [00:48<02:16,  2.88it/s]


 26%|██▌       | 138/530 [00:48<02:03,  3.18it/s]


 26%|██▌       | 139/530 [00:48<02:20,  2.79it/s]


 27%|██▋       | 141/530 [00:49<02:08,  3.02it/s]


 27%|██▋       | 142/530 [00:49<02:11,  2.95it/s]


 27%|██▋       | 143/530 [00:50<02:07,  3.02it/s]


 27%|██▋       | 144/530 [00:50<01:57,  3.29it/s]


 27%|██▋       | 145/530 [00:51<02:25,  2.64it/s]


 28%|██▊       | 146/530 [00:51<02:00,  3.20it/s]


 28%|██▊       | 147/530 [00:51<02:16,  2.80it/s]


 28%|██▊       | 149/530 [00:52<02:15,  2.82it/s]


 28%|██▊       | 150/530 [00:52<02:20,  2.70it/s]


 28%|██▊       | 151/530 [00:52<01:59,  3.17it/s]


 29%|██▊       | 152/530 [00:53<01:56,  3.23it/s]


 29%|██▉    

 55%|█████▍    | 289/530 [01:40<01:30,  2.66it/s]


 55%|█████▍    | 290/530 [01:41<01:23,  2.86it/s]


 55%|█████▍    | 291/530 [01:41<01:35,  2.51it/s]


 55%|█████▌    | 292/530 [01:42<01:25,  2.80it/s]


 55%|█████▌    | 294/530 [01:42<01:18,  2.99it/s]


 56%|█████▌    | 295/530 [01:42<01:17,  3.05it/s]


 56%|█████▌    | 296/530 [01:43<01:14,  3.13it/s]


 56%|█████▌    | 297/530 [01:43<01:32,  2.53it/s]


 56%|█████▌    | 298/530 [01:43<01:13,  3.15it/s]


 56%|█████▋    | 299/530 [01:44<01:33,  2.47it/s]


 57%|█████▋    | 300/530 [01:44<01:20,  2.86it/s]


 57%|█████▋    | 302/530 [01:45<01:16,  2.96it/s]


 57%|█████▋    | 303/530 [01:45<01:14,  3.06it/s]


 57%|█████▋    | 304/530 [01:45<01:09,  3.25it/s]


 58%|█████▊    | 305/530 [01:46<01:24,  2.65it/s]


 58%|█████▊    | 306/530 [01:46<01:16,  2.94it/s]


 58%|█████▊    | 307/530 [01:47<01:23,  2.66it/s]


 58%|█████▊    | 308/530 [01:47<01:07,  3.31it/s]


 58%|█████▊    | 310/530 [01:48<01:22,  2.68it/s]


 59%|█████▉ 

 86%|████████▌ | 455/530 [02:38<00:19,  3.78it/s]


 86%|████████▌ | 456/530 [02:38<00:28,  2.59it/s]


 86%|████████▌ | 457/530 [02:39<00:22,  3.18it/s]


 87%|████████▋ | 459/530 [02:40<00:28,  2.49it/s]


 87%|████████▋ | 460/530 [02:40<00:27,  2.50it/s]


 87%|████████▋ | 463/530 [02:40<00:18,  3.68it/s]


 88%|████████▊ | 464/530 [02:41<00:23,  2.85it/s]


 88%|████████▊ | 466/530 [02:42<00:19,  3.35it/s]


 88%|████████▊ | 467/530 [02:42<00:24,  2.54it/s]


 88%|████████▊ | 468/530 [02:43<00:24,  2.50it/s]


 89%|████████▉ | 471/530 [02:43<00:18,  3.25it/s]


 89%|████████▉ | 472/530 [02:44<00:21,  2.68it/s]


 89%|████████▉ | 473/530 [02:44<00:18,  3.10it/s]


 89%|████████▉ | 474/530 [02:44<00:17,  3.26it/s]


 90%|████████▉ | 475/530 [02:45<00:20,  2.66it/s]


 90%|████████▉ | 476/530 [02:45<00:20,  2.64it/s]


 90%|█████████ | 478/530 [02:46<00:13,  3.91it/s]


 90%|█████████ | 479/530 [02:46<00:16,  3.04it/s]


 91%|█████████ | 480/530 [02:47<00:20,  2.40it/s]


 91%|███████

Runtime: 215.405191 s


...

Multiplets | family 3:   0%|          | 0/55 [00:13<?, ?it/s]

Multiplets | family 5:   0%|          | 0/288 [00:00<?, ?it/s]


  0%|          | 0/288 [00:00<?, ?it/s]


  0%|          | 1/288 [00:05<25:08,  5.26s/it]


  1%|          | 2/288 [00:05<10:47,  2.26s/it]


  1%|          | 3/288 [00:05<06:03,  1.28s/it]


  1%|▏         | 4/288 [00:05<03:51,  1.23it/s]


  2%|▏         | 5/288 [00:05<02:41,  1.76it/s]


  2%|▏         | 6/288 [00:05<02:04,  2.26it/s]


  3%|▎         | 8/288 [00:06<01:13,  3.81it/s]


  3%|▎         | 9/288 [00:10<06:19,  1.36s/it]


  3%|▎         | 10/288 [00:10<05:02,  1.09s/it]


  4%|▍         | 11/288 [00:11<03:49,  1.21it/s]


  5%|▍         | 14/288 [00:11<02:04,  2.20it/s]


  6%|▌         | 16/288 [00:11<01:39,  2.74it/s]


  6%|▌         | 17/288 [00:15<04:55,  1.09s/it]


  6%|▋         | 18/288 [00:16<04:26,  1.01it/s]


  7%|▋         | 20/288 [00:16<02:59,  1.49it/s]


  8%|▊         | 22/288 [00:17<02:05,  2.12it/s]


  8%|▊         | 23/28

 52%|█████▏    | 149/288 [01:49<01:24,  1.65it/s]


 52%|█████▏    | 150/288 [01:50<01:39,  1.39it/s]


 52%|█████▏    | 151/288 [01:50<01:28,  1.55it/s]


 53%|█████▎    | 152/288 [01:51<01:17,  1.76it/s]


 53%|█████▎    | 153/288 [01:52<01:28,  1.53it/s]


 53%|█████▎    | 154/288 [01:53<01:50,  1.21it/s]


 54%|█████▍    | 156/288 [01:53<01:12,  1.81it/s]


 55%|█████▍    | 157/288 [01:54<01:12,  1.80it/s]


 55%|█████▍    | 158/288 [01:55<01:40,  1.29it/s]


 55%|█████▌    | 159/288 [01:56<01:30,  1.43it/s]


 56%|█████▌    | 160/288 [01:57<01:29,  1.42it/s]


 56%|█████▌    | 161/288 [01:57<01:14,  1.70it/s]


 56%|█████▋    | 162/288 [01:58<01:37,  1.29it/s]


 57%|█████▋    | 163/288 [01:58<01:21,  1.54it/s]


 57%|█████▋    | 164/288 [01:59<01:13,  1.69it/s]


 57%|█████▋    | 165/288 [01:59<01:06,  1.85it/s]


 58%|█████▊    | 166/288 [02:01<01:41,  1.21it/s]


 58%|█████▊    | 167/288 [02:01<01:25,  1.41it/s]


 58%|█████▊    | 168/288 [02:02<01:16,  1.56it/s]


 59%|█████▊ 

Multiplets | family 4:   0%|          | 0/268 [00:00<?, ?it/s]



  0%|          | 0/268 [00:00<?, ?it/s]



  0%|          | 1/268 [00:03<15:41,  3.52s/it]



  1%|          | 2/268 [00:03<06:43,  1.52s/it]



  3%|▎         | 7/268 [00:03<01:20,  3.25it/s]



  4%|▎         | 10/268 [00:07<02:47,  1.54it/s]



  4%|▍         | 12/268 [00:07<02:03,  2.07it/s]



  6%|▌         | 15/268 [00:07<01:20,  3.15it/s]



  6%|▋         | 17/268 [00:10<02:47,  1.50it/s]



  7%|▋         | 19/268 [00:10<02:06,  1.97it/s]



  8%|▊         | 21/268 [00:11<01:39,  2.47it/s]



  9%|▊         | 23/268 [00:11<01:15,  3.26it/s]



  9%|▉         | 25/268 [00:14<02:45,  1.47it/s]



 10%|▉         | 26/268 [00:14<02:22,  1.69it/s]



 10%|█         | 27/268 [00:14<02:04,  1.94it/s]



 10%|█         | 28/268 [00:14<01:42,  2.34it/s]



 11%|█         | 29/268 [00:15<01:25,  2.81it/s]



 12%|█▏        | 31/268 [00:15<01:09,  3.41it/s]



 12%|█▏        | 33/268 [00:18<02:40,  1.46it/s]



 13%|█▎   

 57%|█████▋    | 154/268 [01:14<01:03,  1.80it/s]



 58%|█████▊    | 155/268 [01:14<01:11,  1.57it/s]



 58%|█████▊    | 156/268 [01:15<01:07,  1.65it/s]



 59%|█████▊    | 157/268 [01:15<00:54,  2.02it/s]



 59%|█████▉    | 158/268 [01:15<00:43,  2.52it/s]



 59%|█████▉    | 159/268 [01:16<00:39,  2.75it/s]



 60%|█████▉    | 160/268 [01:16<00:31,  3.43it/s]



 60%|██████    | 161/268 [01:16<00:41,  2.55it/s]



 60%|██████    | 162/268 [01:18<01:03,  1.66it/s]



 61%|██████    | 163/268 [01:19<01:18,  1.34it/s]



 61%|██████    | 164/268 [01:19<01:06,  1.57it/s]



 62%|██████▏   | 165/268 [01:19<00:51,  1.98it/s]



 62%|██████▏   | 166/268 [01:19<00:40,  2.49it/s]



 62%|██████▏   | 167/268 [01:20<00:48,  2.09it/s]



 63%|██████▎   | 169/268 [01:21<00:40,  2.47it/s]



 63%|██████▎   | 170/268 [01:22<00:58,  1.68it/s]



 64%|██████▍   | 171/268 [01:23<01:06,  1.46it/s]



 64%|██████▍   | 172/268 [01:23<01:04,  1.48it/s]



 65%|██████▍   | 174/268 [01:24<00:44,  2.10it

Runtime: 331.257738 s


In [68]:
for diagnosis, diag_data in selected_boot_stats.items():
    for order, mults in diag_data.items():
        for mult, distro in mults.items():
            selected_boot_stats[diagnosis][order][mult] = {
                'observed_O' : o_infoptim.o_information(EDI_diags[diagnosis][list(mult)]),
                'samp_distro' : distro
            }

Multiplets | family 3:   0%|          | 0/3 [00:01<?, ?it/s]
Multiplets | family 5:   0%|          | 0/288 [05:35<?, ?it/s]
Multiplets | family 4:   0%|          | 0/268 [02:16<?, ?it/s]


In [69]:
bca_final = {}
for diagnosis, mult_scales in selected_boot_stats.items():
    start = time.perf_counter()
    # structure: order: multiple: samp_distro
    bca_final[diagnosis] = bca_bootopt.bca_bootstrap_parallel(multiplets_dict=mult_scales, 
                                                                            X=EDI_diags[diagnosis])
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")

Families:   0%|          | 0/3 [00:00<?, ?it/s]


[BCa | family 5] starting – 52 multiplets
  [BCa | family 5] 52/52 multiplets (100%)

[BCa | family 4] starting – 530 multiplets


Families: 100%|██████████| 3/3 [00:00<00:00,  9.34it/s]


  [BCa | family 4] 530/530 multiplets (100%)

[BCa | family 3] starting – 21 multiplets
  [BCa | family 3] 21/21 multiplets (100%)
Runtime: 0.321626 s


Families:  33%|███▎      | 1/3 [00:00<00:00,  6.49it/s]

...

  [BCa | family 4] 474/474 multiplets (100%)

[BCa | family 3] starting – 55 multiplets
  [BCa | family 3] 55/55 multiplets (100%)
Runtime: 0.277762 s


Families:  33%|███▎      | 1/3 [00:00<00:00,  7.84it/s]


[BCa | family 5] starting – 288 multiplets
  [BCa | family 5] 288/288 multiplets (100%)

[BCa | family 4] starting – 268 multiplets


Families: 100%|██████████| 3/3 [00:00<00:00, 10.80it/s]

  [BCa | family 4] 268/268 multiplets (100%)

[BCa | family 3] starting – 3 multiplets
  [BCa | family 3] 3/3 multiplets (100%)
Runtime: 0.281130 s


#### Hierarchical Pruning: actually filtering out the overlapping multiplets

In [70]:
filtered_D = {} 
discarded_D = {}
for diagnose, orders in bca_final.items():
    filtered, discarded = sel.select_multiplets_minimal(orders)
    filtered_D[diagnose] = filtered
    discarded_D[diagnose] = discarded

### POLY - FINAL STEP ( $\Omega$ - DF)

In [71]:
sel_finale = {}
for diagnosis, sel_mults in filtered_D.items():
    sel_finale[diagnosis] = pp_bootopt.compute_omega_for_multiplets(multiplets_dict=sel_mults, 
                                                                                   X=EDI_diags[diagnosis])
    ap = sel_finale[diagnosis]
    sel_finale[diagnosis]["label"] = np.where(ap["omega"] < 0, "synergy", "redundancy")

In [72]:
sel_finale['ANR'].head()

,order,label,omega,node1,node2,node3,node4,node5
0,3,redundancy,0.135900,2,7,16,<NA>,<NA>
1,3,redundancy,0.185974,2,9,32,<NA>,<NA>
2,3,redundancy,0.207371,2,16,32,<NA>,<NA>
3,3,redundancy,0.166395,2,32,49,<NA>,<NA>
4,3,redundancy,0.178320,7,9,32,<NA>,<NA>


In [73]:
# skip dir creation if it already exist
out_dir_sm = "selected_he_POLY"            
os.makedirs(out_dir_sm, exist_ok=True)

for diagnosis, df in sel_finale.items():
    fname = f"{diagnosis}_selected_he_POLY.csv"
    fpath = os.path.join(out_dir_sm, fname)
    df.to_csv(fpath, index=False)
# then move the files to DATA/selected_HYPEREDGES_data/POLY folder.

### Partition and Multiplex Construction 

**After all three validation stages:**

**Partition by sign:**
   - **Redundant hyperedges:** Ω > 0 (positive layer)
   - **Synergistic hyperedges:** Ω < 0 (negative layer)
   
**Final Output:** 
- Conservative set of statistically significant, stable, and genuinely higher-order multiplets